# Assistente de Cobrança — Demo (Prova de Conceito) · v3 (cenários de call center, Claude ou GPT)

**Objetivo:** um chat interno em que o **operador de cobrança** pergunta sobre um produto **ou sobre uma situação de atendimento** (prazo do acordo, parcelamento, desconto, custas processuais, quebra de acordo...) e recebe, em linguagem de atendimento, a resposta baseada **somente** na documentação do Sicoob (deck "Produtos Cobrança") e nas regras de simulação deste notebook.

**Como funciona (resumo para os gestores):**

```
Pergunta do operador
   │
   ▼
[1] Roteadores (palavras-chave, custo zero)
      ├─ de PRODUTO  → cartão, consignado, prejuízo...
      └─ de CENÁRIO  → prazo, parcelamento, custas, quebra de acordo...
   │
   ▼
[2] Monta contexto enxuto: ficha do produto + política de acordo + regras do cenário + glossário
   │
   ▼
[3] Modelo de linguagem pequeno responde só com esse contexto
   │
   ▼
Resposta curta + sugestão de fala ao cliente
```

**Decisões de projeto**
- **Sem busca vetorial (RAG) nesta demo:** a base é pequena e estruturada. Enviamos só o que importa para a pergunta: menos tokens, menos risco de trazer o produto errado.
- **Tokens baixos:** contexto enxuto, histórico curto e limite de resposta.
- **Tudo que é política de acordo, prazo, parcelamento, desconto e jurídico é FICTÍCIO.** O PDF não traz essas regras e elas variam por cooperativa. Os dados do PDF (fichas de produto, regra "pode/não pode informar") são os reais.
- **Sem dados reais de clientes** (LGPD).

## Célula 1 — Instalação

**O que fazer:** rodar a célula abaixo e esperar terminar **antes** de rodar a próxima.

**O que explicar:** instalamos três bibliotecas: `anthropic` e `openai` (comunicação com o modelo de linguagem; o notebook usa a que tiver chave) e `gradio` (cria a página de chat com link para o gestor testar).

In [ ]:
!pip -q install anthropic openai gradio

## Célula 2 — Chave de API e teste de conexão

**O que fazer:**
1. No Colab, clique no ícone de **chave (Secrets)** na barra lateral esquerda.
2. Tenha **um** destes segredos, com **"Acesso ao notebook"** ativado:
   - `ANTHROPIC_API_KEY`: chave da Anthropic (usa Claude), ou
   - `GPT` (ou `OPENAI_API_KEY`): chave da OpenAI (usa GPT).
3. Rode a célula. Se existirem os dois, ela usa o Claude.

**O que explicar:** a chave **nunca** fica escrita no código. A função `chamar_modelo` é a **única** parte do notebook que conhece o fornecedor do modelo. Fichas, roteadores, cenários e testes não mudam se trocarmos de Claude para GPT (ou o contrário). `MAX_TOKENS_RESPOSTA` limita o tamanho da resposta, o que controla custo. O teste final confirma a conexão e mostra a contagem de tokens, a unidade de cobrança.

In [ ]:
import os
from google.colab import userdata


def _segredo(nome):
    # devolve o valor do segredo ou None se ele não existir / não tiver acesso ao notebook
    try:
        return userdata.get(nome)
    except Exception:
        return None


CHAVE_ANTHROPIC = _segredo("ANTHROPIC_API_KEY")
CHAVE_OPENAI = _segredo("GPT") or _segredo("OPENAI_API_KEY")

if CHAVE_ANTHROPIC:
    import anthropic
    PROVEDOR = "anthropic"
    client = anthropic.Anthropic(api_key=CHAVE_ANTHROPIC)
    MODELO = "claude-haiku-5-5"      # modelo pequeno e barato (confira o nome atual na documentação)
elif CHAVE_OPENAI:
    from openai import OpenAI
    PROVEDOR = "openai"
    client = OpenAI(api_key=CHAVE_OPENAI)
    MODELO = "gpt-4o-mini"           # modelo pequeno e barato
else:
    raise RuntimeError(
        "Nenhuma chave encontrada. Em Secrets (ícone de chave), crie ANTHROPIC_API_KEY ou GPT "
        "e ative 'Acesso ao notebook'."
    )

MAX_TOKENS_RESPOSTA = 400        # teto da resposta -> controla custo
MAX_HISTORICO = 6                # quantas mensagens anteriores enviamos


def chamar_modelo(system, mensagens, max_tokens):
    # único ponto de contato com o modelo. Retorna (texto, tokens_entrada, tokens_saida)
    if PROVEDOR == "anthropic":
        r = client.messages.create(model=MODELO, max_tokens=max_tokens, system=system, messages=mensagens)
        return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens
    r = client.chat.completions.create(
        model=MODELO, max_completion_tokens=max_tokens, temperature=0,
        messages=[{"role": "system", "content": system}] + mensagens,
    )
    return r.choices[0].message.content, r.usage.prompt_tokens, r.usage.completion_tokens


texto, t_in, t_out = chamar_modelo("Responda de forma curta.", [{"role": "user", "content": "Responda apenas: ok"}], 20)
print("Provedor:", PROVEDOR, "| Modelo:", MODELO)
print("Resposta:", texto)
print("Tokens -> entrada:", t_in, "| saída:", t_out)

## Célula 3 — Base de conhecimento: fichas de produto

**O que fazer:** rodar a célula (o conteúdo vem do PDF, resumido em fichas).

**O que explicar:** é o "cérebro" do chat. Cada produto do deck virou uma **ficha estruturada**, com nome, categoria, **regra de atualização de valor** (D+0, D-2, fechamento da fatura), descrição, pontos de atenção e `keywords` (palavras que o roteador usa para achar a ficha).

Por que fichas e não fatiar o PDF em pedaços de 300 caracteres? Porque **cada slide já é uma unidade completa de sentido**; fatiar poderia separar o produto da sua regra.

> Para alimentar o chat com novos produtos, basta adicionar uma ficha aqui, sem mexer no resto.

In [ ]:
FICHAS = {
    "cartao": {
        "nome": "Cartão (crédito e/ou débito)",
        "categoria": "Cartões",
        "atualizacao": "Atualizado conforme o último fechamento da fatura",
        "descricao": "Instrumento de pagamento, físico ou virtual, emitido e concedido pelo emissor, de uso pessoal e intransferível do portador, com função de crédito e/ou débito. Contém número, características de segurança, nome do portador, prazo de validade e identificação da bandeira e do emissor.",
        "atencao": [],
        "keywords": ["cartao", "cartoes", "fatura", "bandeira"],
    },
    "titulos_descontados": {
        "nome": "Títulos Descontados (TD)",
        "categoria": "Cobrança / Desconto de títulos",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Funciona como adiantamento de capital: o banco antecipa o valor da duplicata mediante cobrança de uma taxa. É o adiantamento de um dinheiro que entraria no caixa da empresa no futuro, tendo o título como garantia.",
        "atencao": ["No dia do vencimento, se o título não for pago, a responsabilidade pela quitação é do cooperado que adiantou o título."],
        "keywords": ["titulo descontado", "titulos descontados", "td", "duplicata", "desconto de titulo", "desconto de titulos", "desconto de duplicata"],
    },
    "credito_rural": {
        "nome": "Crédito Rural",
        "categoria": "Agronegócio",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Módulo em que o associado gerencia propostas de linhas e programas de crédito rural, que incentivam o desenvolvimento agropecuário e a manutenção do pequeno ao grande produtor. As linhas se dividem por finalidade: CUSTEIO (despesas normais dos ciclos produtivos agrícolas e pecuários, como silos, bags e insumos); INVESTIMENTO (serviços ou bens que se estendem por vários períodos de produção, para implantação, modernização e expansão da agropecuária); COMERCIALIZAÇÃO (despesas de comercialização do produto ou conversão em espécie dos títulos oriundos da venda e entrega às empresas beneficiadoras).",
        "atencao": ["As operações podem ou não estar vinculadas aos programas: Pronaf (Programa Nacional de Fortalecimento da Agricultura Familiar, MCR 10), Pronamp (Programa Nacional de Apoio ao Médio Produtor Rural, MCR 8) e Funcafé (Fundo de Defesa da Economia Cafeeira, MCR 9)."],
        "keywords": ["credito rural", "rural", "pronaf", "pronamp", "funcafe", "custeio", "agricultura familiar", "produtor rural", "comercializacao"],
    },
    "tarifas": {
        "nome": "Conta Corrente — Tarifas",
        "categoria": "Conta Corrente",
        "atualizacao": "Valor atualizado D-2, sem acréscimo de juros e encargos",
        "descricao": "Valores cobrados pela prestação de um serviço, definidos livremente por cada instituição. A cobrança deve estar prevista no contrato entre instituição e cliente, ou o serviço deve ter sido previamente autorizado ou solicitado pelo cliente.",
        "atencao": ["A regulamentação em vigor (Resolução 3.919 do BACEN) classifica os serviços prestados a pessoas físicas em Essenciais, Prioritários, Especiais e Diferenciados."],
        "keywords": ["tarifa", "tarifas", "resolucao 3919", "servicos essenciais", "cobranca de tarifa"],
    },
    "cheque_especial": {
        "nome": "Conta Corrente — Cheque Especial",
        "categoria": "Conta Corrente",
        "atualizacao": "Valor atualizado D-2, sem acréscimo de juros e encargos",
        "descricao": "É um 'Limite de Crédito Rotativo' (valor monetário) pré-aprovado, vinculado a uma conta corrente. A Cooperativa coloca à disposição do associado um valor determinado, usado automaticamente caso o dinheiro disponível na conta seja insuficiente para honrar os débitos.",
        "atencao": [],
        "keywords": ["cheque especial", "limite da conta", "limite de credito rotativo"],
    },
    "adiantamento_depositante": {
        "nome": "Conta Corrente — Adiantamento a Depositante",
        "categoria": "Conta Corrente",
        "atualizacao": "Valor atualizado D-2, sem acréscimo de juros e encargos",
        "descricao": "Serviço de conveniência para fazer uma transação quando não há saldo em conta ou quando o limite de cheque especial já foi usado.",
        "atencao": ["A tarifa é cobrada toda vez que a instituição adianta um valor que não existe na conta no momento do pagamento ou saque. O valor é cobrado no mês seguinte e a taxa fixa consta na Tabela Geral de Tarifas do Banco."],
        "keywords": ["adiantamento a depositante", "adiantamento depositante", "depositante", "conta estourada", "sem saldo"],
    },
    "emprestimo_geral": {
        "nome": "Empréstimo (visão geral)",
        "categoria": "Empréstimo",
        "atualizacao": "Empréstimos atualizados em D+0",
        "descricao": "Contrato pelo qual se toma ou concede certa quantia de dinheiro, com a pressuposição de que será devolvida ao emprestador, com ou sem acréscimo de juros. Tipos: Antecipação de IR, Antecipação de 13º, Crédito Pessoal, Crédito Pré-aprovado, Crédito com Garantia de Imóvel Residencial, Crédito Rotativo, Crédito Consignado e Crédito Imobiliário.",
        "atencao": [],
        "keywords": ["emprestimo", "emprestimos"],
    },
    "antecipacao_ir": {
        "nome": "Empréstimo — Antecipação de IR",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Empréstimo para antecipar o valor da restituição do Imposto de Renda de associados/clientes pessoas físicas que indicaram entidade do Sicoob para receber a restituição.",
        "atencao": ["Beneficiário: ser associado da cooperativa ou correntista do Bancoob; ser titular de conta corrente ativa em cooperativa singular ou na agência do Bancoob (proibido usar conta salário para obter empréstimos); ter indicado a conta corrente da entidade Sicoob de origem da operação para o crédito da restituição do IRPF na declaração enviada à Receita Federal."],
        "keywords": ["antecipacao de ir", "antecipacao do ir", "imposto de renda", "restituicao", "irpf"],
    },
    "antecipacao_13": {
        "nome": "Empréstimo — Antecipação de 13º",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Antecipa o valor do 13º salário de associados/clientes que recebem salário na cooperativa singular ou no Bancoob.",
        "atencao": ["Beneficiário: ser associado da cooperativa singular ou correntista do Bancoob; ser assalariado; ser titular de conta corrente ativa (proibido usar conta-salário para obter empréstimos); receber salário por meio da entidade credora da operação."],
        "keywords": ["antecipacao de 13o", "antecipacao do 13o", "13o", "decimo terceiro", "13 salario"],
    },
    "credito_pessoal": {
        "nome": "Empréstimo — Crédito Pessoal",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Atende necessidades financeiras de associados/clientes pessoas físicas, sem comprovação da destinação do recurso. É proibida a vinculação de imóvel residencial como garantia nesta linha.",
        "atencao": ["Crédito Pessoal Price: prestações iguais, calculadas pela Tabela Price.", "Crédito Pessoal SACD: prestações decrescentes, calculadas pelo Sistema de Amortização Constante Decrescente (nome conforme o material)."],
        "keywords": ["credito pessoal", "pessoal price", "pessoal sacd", "price", "sacd"],
    },
    "credito_garantia_imovel": {
        "nome": "Empréstimo — Crédito com Garantia de Imóvel Residencial",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Crédito pessoal sem destinação específica, direcionado a pessoa física, com garantia de imóvel residencial.",
        "atencao": ["Linha Price: prestações iguais (Tabela Price).", "Linha SACD: prestações decrescentes (Sistema de Amortização Constante Decrescente, nome conforme o material)."],
        "keywords": ["garantia de imovel", "garantia de imovel residencial", "imovel como garantia", "imovel em garantia"],
    },
    "credito_preaprovado": {
        "nome": "Empréstimo — Crédito Pré-aprovado",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Modalidade disponibilizada aos associados/clientes pelos canais eletrônicos, para atender emergências financeiras, sem comprovação da destinação dos recursos.",
        "atencao": ["O aceite eletrônico do Termo de Adesão está disponível no SicoobNet Pessoal, Empresarial e SicoobNet Celular. O aceite manual está disponível no atendimento presencial das entidades."],
        "keywords": ["pre aprovado", "credito pre aprovado", "sicoobnet", "termo de adesao"],
    },
    "credito_consignado": {
        "nome": "Empréstimo — Crédito Consignado",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Empréstimo pessoal sem destinação específica, com débito das parcelas por consignação em folha de pagamento.",
        "atencao": ["Público-alvo: funcionários de instituições públicas e empresas privadas conveniadas, aposentados e pensionistas do INSS."],
        "keywords": ["consignado", "credito consignado", "folha de pagamento", "consignacao"],
    },
    "credito_rotativo": {
        "nome": "Empréstimo — Crédito Rotativo",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Linha de crédito aberta para pessoa física ou empresa com limite predefinido, que pode ser usada automaticamente pelo tomador conforme a necessidade. Pode ser vista como um empréstimo emergencial, semelhante ao cheque especial.",
        "atencao": ["O crédito disponível diminui à medida que o tomador usa e aumenta conforme ele paga o valor principal já usado."],
        "keywords": ["credito rotativo", "rotativo"],
    },
    "capital_giro": {
        "nome": "Empréstimo — Capital de Giro (pessoa jurídica)",
        "categoria": "Empréstimo",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "Empréstimo para pessoa jurídica, vinculado à necessidade de capital de giro das empresas. Separado por data de vencimento: Capital de Giro até 365 dias e acima de 365 dias.",
        "atencao": ["Capital de Giro 13º: empréstimo para a empresa pagar o 13º salário dos empregados."],
        "keywords": ["capital de giro", "giro", "capital de giro 13o"],
    },
    "financiamento_bens_servicos": {
        "nome": "Financiamento — Bens e Serviços",
        "categoria": "Financiamento",
        "atualizacao": "Regra de atualização não informada no material",
        "descricao": "Crédito para pessoa física ou jurídica financiar bens e serviços. Exige comprovação prévia da destinação do recurso, por orçamento que descreva os bens a financiar. Outros tipos: Veículos Novos e Veículos Usados (também exigem orçamento com descrição dos bens).",
        "atencao": [],
        "keywords": ["financiamento", "financiamentos", "bens e servicos", "veiculo", "veiculos", "veiculos novos", "veiculos usados", "carro", "moto"],
    },
    "credito_imobiliario": {
        "nome": "Financiamento — Crédito Imobiliário",
        "categoria": "Financiamento",
        "atualizacao": "Valor atualizado D+0",
        "descricao": "O Bancoob disponibiliza aos associados pessoas físicas o Sicoob Crédito Imobiliário, inicialmente pela linha de financiamento no SFH, para adquirir imóvel residencial novo ou usado, concluído, com ou sem intermediação de recursos do FGTS para compor o valor financiado.",
        "atencao": ["É possível compor renda com até 3 proponentes. O proponente principal (o de maior renda) deve ser associado da cooperativa; os demais são cadastrados como clientes."],
        "keywords": ["credito imobiliario", "imobiliario", "sfh", "fgts", "imovel residencial", "casa propria"],
    },
    "honra_aval": {
        "nome": "Honras de Avais (produto bancário)",
        "categoria": "Produto bancário",
        "atualizacao": "Não informado no material",
        "descricao": "Ocorrência de produto bancário com regra específica sobre o que pode ser informado em contato de cobrança.",
        "atencao": [],
        "keywords": ["honra de aval", "honra de avais", "honras de avais", "honra"],
    },
    "prejuizo": {
        "nome": "Prejuízo (produto bancário)",
        "categoria": "Produto bancário",
        "atualizacao": "Não informado no material",
        "descricao": "Ocorrência de produto bancário com regra específica sobre o que pode ser informado em contato de cobrança.",
        "atencao": [],
        "keywords": ["prejuizo", "prejuizos", "baixado para prejuizo"],
    },
    "avalista": {
        "nome": "Avalista (2º responsável pelo pagamento)",
        "categoria": "Garantias",
        "atualizacao": "Não se aplica",
        "descricao": "Avalista é quem aceita ser responsável pelo pagamento de empréstimo ou financiamento realizado por outra pessoa. Quem dá o aval é garantidor da dívida e assume o risco de, se o devedor não pagar, responder pelo pagamento, inclusive com todos os seus bens.",
        "atencao": [],
        "keywords": ["avalista", "avalistas", "aval", "garantidor", "segundo responsavel"],
    },
}

print(f"{len(FICHAS)} fichas carregadas.")

## Célula 4 — Regras de cobrança, política de acordos (FICTÍCIA) e glossário

**O que fazer:** rodar a célula.

**O que explicar:** três blocos, separados das fichas de propósito:

1. **`REGRAS_INFORMAR`**: regra do próprio PDF. Em *Honras de Avais* e *Prejuízo*, o operador **pode informar** vencimento, dias de atraso e contrato, e **não pode informar** valores nem produto.
2. **`POLITICA_ACORDOS`**: agora **estruturada por campo**, para o chat responder "pode parcelar?", "qual o prazo?" e "quanto de desconto?" com precisão: `permite_acordo`, máximo de parcelas, entrada mínima, desconto máximo, prazo para a 1ª parcela e quem aprova. **Tudo fictício.** Em produção a empresa preenche (por cooperativa) sem mexer no código.
3. **`GLOSSARIO`**: termos do material (D+0, D-2, Price, SACD, avalista).

As funções `formatar_politica` e `resumo_politica` transformam esses campos em texto curto para o prompt. O resumo é usado quando o operador pergunta algo geral, sem citar produto.

In [ ]:
# 1) Regras do PDF: o que pode / não pode ser informado
REGRAS_INFORMAR = {
    "honra_aval": {
        "pode": ["Vencimento", "Dias de atraso", "Contrato"],
        "nao_pode": ["Valores", "Produto"],
    },
    "prejuizo": {
        "pode": ["Vencimento", "Dias de atraso", "Contrato"],
        "nao_pode": ["Valores", "Produto"],
    },
}

# 2) Política de acordos — DADOS FICTÍCIOS (a empresa preencherá os reais por cooperativa)
POLITICA_PADRAO = {"permite_acordo": "não cadastrado", "obs": "Sem política cadastrada. Acionar a gestão.", "aprovador": "Gestão"}


def pol(permite="sim", parcelas=None, entrada=None, desconto=None, dias=None, aprovador="Gestão", obs=""):
    return {"permite_acordo": permite, "parcelas_max": parcelas, "entrada_min_pct": entrada,
            "desconto_max_pct": desconto, "primeira_parcela_dias": dias, "aprovador": aprovador, "obs": obs}


POLITICA_ACORDOS = {
    "cartao": pol(parcelas=12, entrada=10, desconto=20, dias=10,
                  aprovador="Operador até 10% de desconto; acima disso, gestão",
                  obs="Valor conforme o último fechamento da fatura."),
    "titulos_descontados": pol(parcelas=6, entrada=20, desconto=15, dias=5,
                               obs="Avaliar reapresentação do título antes de renegociar."),
    "credito_rural": pol(obs="Prorrogação conforme normativo do programa e análise da cooperativa; sem parcelamento livre."),
    "tarifas": pol("não", obs="Tarifa não é negociada em cobrança; cobrança indevida segue para o atendimento.", aprovador="Não se aplica"),
    "cheque_especial": pol(parcelas=24, entrada=0, desconto=20, dias=7,
                           aprovador="Operador até 10% de desconto; acima disso, gestão",
                           obs="Conversão do saldo devedor em crédito pessoal parcelado."),
    "adiantamento_depositante": pol(parcelas=6, entrada=0, desconto=10, dias=5, aprovador="Operador",
                                    obs="Regularização do saldo da conta."),
    "credito_pessoal": pol(parcelas=48, entrada=10, desconto=30, dias=7,
                           obs="Desconto de 30% só para atraso acima de 90 dias; até 90 dias, no máximo 10%."),
    "credito_garantia_imovel": pol(parcelas=60, entrada=15, desconto=15, dias=10,
                                   obs="Renegociação com manutenção da garantia."),
    "credito_preaprovado": pol(parcelas=12, entrada=10, desconto=10, dias=7, aprovador="Operador"),
    "credito_consignado": pol("não", obs="Desconto em folha; tratar com o convênio/consignante.", aprovador="Não se aplica"),
    "credito_rotativo": pol(parcelas=12, entrada=0, desconto=15, dias=5, aprovador="Operador"),
    "capital_giro": pol(parcelas=36, entrada=20, desconto=20, dias=10,
                        obs="Renegociação de prazo e parcelas mediante análise da cooperativa."),
    "financiamento_bens_servicos": pol(parcelas=36, entrada=15, desconto=10, dias=10,
                                       obs="Verificar a situação do bem financiado antes de renegociar."),
    "credito_imobiliario": pol(parcelas=60, entrada=10, desconto=15, dias=10,
                               obs="Renegociação conforme regras do financiamento (SFH); análise individual."),
    "honra_aval": pol(parcelas=24, entrada=20, desconto=20, dias=7,
                      obs="Respeitar a regra de informação (pode/não pode informar)."),
    "prejuizo": pol(parcelas=48, entrada=10, desconto=50, dias=7,
                    obs="Desconto maior permitido em prejuízo, sempre com aprovação. Respeitar a regra de informação."),
    "avalista": pol(obs="Segue as condições do acordo do contrato principal."),
}


def formatar_politica(p):
    if p.get("permite_acordo") != "sim":
        return f"permite acordo: {p.get('permite_acordo')}. {p.get('obs', '')} Aprovação: {p.get('aprovador', 'Gestão')}."
    itens = ["permite acordo: sim"]
    if p.get("parcelas_max"):
        itens.append(f"parcelamento em até {p['parcelas_max']}x")
    else:
        itens.append("parcelamento: sem número fixo (análise da gestão)")
    if p.get("entrada_min_pct") is not None:
        itens.append(f"entrada mínima de {p['entrada_min_pct']}%")
    if p.get("desconto_max_pct"):
        itens.append(f"desconto máximo de {p['desconto_max_pct']}% sobre juros e multa")
    if p.get("primeira_parcela_dias"):
        itens.append(f"1ª parcela/entrada em até {p['primeira_parcela_dias']} dias")
    itens.append(f"aprovação: {p.get('aprovador', 'Gestão')}")
    texto = "; ".join(itens) + "."
    if p.get("obs"):
        texto += " Obs.: " + p["obs"]
    return texto


def resumo_politica():
    # visão compacta de todos os produtos, usada quando a pergunta não cita produto
    return "\n".join(f"- {FICHAS[pid]['nome']}: {formatar_politica(p)}" for pid, p in POLITICA_ACORDOS.items())


# 3) Glossário (termos que aparecem no material)
GLOSSARIO = {
    "d+0":  {"definicao": "Valor atualizado no próprio dia (D+0).", "keywords": ["d 0", "d+0"]},
    "d-2":  {"definicao": "Valor atualizado com referência D-2, sem acréscimo de juros e encargos.", "keywords": ["d 2", "d-2"]},
    "price": {"definicao": "Sistema de Amortização Tabela Price: prestações iguais.", "keywords": ["price"]},
    "sacd": {"definicao": "Sistema de Amortização Constante Decrescente (nome conforme o material): prestações decrescentes.", "keywords": ["sacd"]},
    "avalista": {"definicao": "Garantidor da dívida: se o devedor não pagar, responde pelo pagamento, inclusive com seus bens.", "keywords": ["avalista", "aval"]},
}

print("Regras, política fictícia estruturada e glossário carregados.")
print()
print("Exemplo -> Cartão:", formatar_politica(POLITICA_ACORDOS["cartao"]))

## Célula 5 — Roteador: descobre o produto da pergunta (custo zero)

**O que fazer:** rodar a célula.

**O que explicar:** antes de chamar o modelo, um código simples procura **palavras-chave** na pergunta (ignora acento e maiúscula) e escolhe a ficha certa. **Não gasta nenhum token.** Palavras mais longas e específicas pesam mais ("capital de giro" vale mais que "13º"). Se dois produtos empatam (ex.: Price/SACD existe em dois), os dois entram no contexto.

É a versão simples e previsível do que, no RAG, seria a busca vetorial.

In [ ]:
import re
import unicodedata


def normalizar(texto):
    # minúsculas, sem acento, só letras/números separados por espaço
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    return re.sub(r"[^a-z0-9]+", " ", texto).strip()


def tem_palavra(texto_norm, kw):
    kw_norm = normalizar(kw)
    return re.search(r"\b" + re.escape(kw_norm) + r"\b", texto_norm) is not None


def rotear_keywords(pergunta, limite=2):
    # retorna lista de ids de produtos, do mais para o menos provável
    texto = normalizar(pergunta)
    pontos = {}
    for pid, ficha in FICHAS.items():
        score = sum(len(normalizar(kw)) for kw in ficha["keywords"] if tem_palavra(texto, kw))
        if score > 0:
            pontos[pid] = score
    if not pontos:
        return []
    ordenados = sorted(pontos, key=pontos.get, reverse=True)
    melhor = pontos[ordenados[0]]
    return [p for p in ordenados if pontos[p] >= 0.6 * melhor][:limite]


print(rotear_keywords("Como funciona a antecipação do 13º?"))

## Célula 6 — Cenários de atendimento (call center), tudo FICTÍCIO

**O que fazer:** rodar a célula.

**O que explicar:** o operador nem sempre pergunta por um produto: muitas vezes pergunta por uma **situação** ("qual o prazo do acordo?", "tem custas processuais?", "o cliente quebrou o acordo"). Por isso criamos uma segunda base, `TEMAS`, com uma regra curta por cenário:

| Cenário | Exemplo de pergunta |
|---|---|
| Prazo do acordo | "Qual o prazo para pagar a entrada?" |
| Parcelamento | "Posso parcelar em 24x?" |
| Desconto | "Quanto de desconto eu posso dar?" |
| Jurídico e custas processuais | "O contrato está no jurídico, quem paga as custas?" |
| Restrição cadastral | "Quando tira o nome do Serasa?" |
| Quebra de acordo | "O cliente atrasou duas parcelas do acordo" |
| Objeção do cliente | "Ele diz que não tem condição de pagar" |
| Formalização e pagamento | "Pode ser pelo Pix?" |

O funcionamento é o mesmo do roteador de produtos (palavras-chave, custo zero). **Produto e cenário se combinam:** "Cartão: posso parcelar em 18x?" aciona a ficha do cartão (com a política dele) **e** a regra de parcelamento.

> **Atenção:** o PDF não traz nada sobre jurídico, prazos ou custas. Todo o conteúdo abaixo é **exemplo inventado para a demo**, e o chat é instruído a tratá-lo assim. Em produção, a área jurídica e a gestão fornecem as regras reais.

In [ ]:
# CENÁRIOS DE ATENDIMENTO — EXEMPLOS FICTÍCIOS (não constam no PDF)
TEMAS = {
    "prazo_acordo": {
        "nome": "Prazo do acordo",
        "regra": "O prazo total de pagamento é o número máximo de parcelas da política do produto, contado a partir da 1ª parcela. A entrada ou 1ª parcela deve ser paga no prazo (em dias) definido na política do produto. A proposta de acordo vale por 3 dias úteis; depois disso precisa ser recalculada. O acordo só é considerado fechado após o pagamento da entrada ou da 1ª parcela.",
        "usa_tabela": True,
        "keywords": ["prazo do acordo", "prazo para pagar", "prazo de pagamento", "validade da proposta", "primeira parcela", "1a parcela", "prazo"],
    },
    "parcelamento": {
        "nome": "Parcelamento",
        "regra": "Só parcele se a política do produto permitir acordo, respeitando o máximo de parcelas e a entrada mínima. A parcela mínima é de R$ 50,00. Os juros do parcelamento seguem a tabela da cooperativa, que não consta no material: não informe taxa e encaminhe à gestão. Se o cliente pedir mais parcelas que o máximo, não prometa: explique o limite e encaminhe à gestão.",
        "usa_tabela": True,
        "keywords": ["parcelar", "parcelamento", "parcelas", "parcela", "entrada", "em quantas vezes", "dividir", "vezes"],
    },
    "desconto": {
        "nome": "Desconto no acordo",
        "regra": "Respeite o desconto máximo da política do produto e a alçada de aprovação. O desconto incide apenas sobre juros e multa, nunca sobre o valor principal. Acima da alçada do operador, o pedido vai para a gestão. Não prometa desconto antes da aprovação.",
        "usa_tabela": True,
        "keywords": ["desconto", "abatimento", "tirar juros", "tirar multa", "reduzir juros", "reduzir multa"],
    },
    "juridico_custas": {
        "nome": "Cobrança judicial e custas processuais",
        "regra": "O material não traz regras jurídicas; este conteúdo é exemplo. Se o contrato estiver em cobrança judicial, o acordo é conduzido pelo jurídico, não pelo operador. O operador não calcula nem informa valores de custas processuais ou honorários advocatícios e não promete isenção ou desconto sobre eles. Custas já pagas e honorários podem ser incluídos no valor do acordo, conforme definição do jurídico da cooperativa. A baixa do processo ocorre após a quitação e a homologação, e o prazo é informado pelo jurídico. Em qualquer dúvida jurídica, encaminhar ao jurídico/gestão.",
        "usa_tabela": False,
        "keywords": ["custas", "custas processuais", "honorarios", "judicial", "juridico", "processo", "advogado", "execucao", "acao judicial"],
    },
    "negativacao": {
        "nome": "Restrição cadastral (Serasa/SPC)",
        "regra": "Após a confirmação do pagamento da entrada ou da 1ª parcela, a baixa da restrição é solicitada em até 5 dias úteis. O prazo final depende do órgão de proteção ao crédito: não prometa data. Em acordo parcelado, a baixa segue a definição da cooperativa.",
        "usa_tabela": False,
        "keywords": ["serasa", "spc", "negativacao", "negativado", "restricao", "nome sujo", "tira o nome", "tirar o nome", "limpar o nome"],
    },
    "quebra_acordo": {
        "nome": "Quebra de acordo",
        "regra": "Há tolerância de 5 dias de atraso na parcela do acordo. Com 2 parcelas em atraso, o acordo é cancelado: a dívida volta ao valor original e os descontos concedidos são perdidos. Um novo acordo precisa de aprovação da gestão.",
        "usa_tabela": False,
        "keywords": ["quebra de acordo", "quebrou o acordo", "quebrou acordo", "descumpriu", "acordo cancelado", "atrasou a parcela do acordo", "parcela do acordo atrasada", "reativar acordo"],
    },
    "objecoes": {
        "nome": "Cliente sem condição de pagar",
        "regra": "Ouça, reconheça a dificuldade e não pressione nem ameace. Alternativas dentro da política do produto: entrada menor (respeitando o mínimo), mais parcelas (até o máximo) ou nova data para a 1ª parcela (até o limite de dias). Se nada couber na política, encaminhe à gestão e registre a tentativa. Nunca prometa condição fora da política.",
        "usa_tabela": True,
        "keywords": ["nao tem dinheiro", "nao tenho dinheiro", "sem dinheiro", "sem condicoes", "nao tem condicao", "nao tem condicoes", "nao tenho condicao", "nao consigo pagar", "nao posso pagar", "desempregado", "dificuldade financeira"],
    },
    "formalizacao": {
        "nome": "Formalização e forma de pagamento",
        "regra": "O acordo é formalizado por termo de acordo e pago por boleto ou Pix emitido pela cooperativa, nunca na conta pessoal de um colaborador. O comprovante deve ser conferido antes de dar o acordo como fechado.",
        "usa_tabela": False,
        "keywords": ["pix", "boleto", "comprovante", "formalizar", "termo de acordo", "pagar o acordo"],
    },
}


def rotear_temas(pergunta, limite=2):
    # mesma lógica do roteador de produtos, aplicada aos cenários de atendimento
    texto = normalizar(pergunta)
    pontos = {}
    for tid, t in TEMAS.items():
        score = sum(len(normalizar(kw)) for kw in t["keywords"] if tem_palavra(texto, kw))
        if score > 0:
            pontos[tid] = score
    if not pontos:
        return []
    ordenados = sorted(pontos, key=pontos.get, reverse=True)
    melhor = pontos[ordenados[0]]
    return [t for t in ordenados if pontos[t] >= 0.6 * melhor][:limite]


print(f"{len(TEMAS)} cenários carregados.")
print(rotear_temas("Posso parcelar o cartão em 18 vezes?"))

## Célula 7 — Teste dos roteadores (grátis)

**O que fazer:** rodar e ver a taxa de acerto dos dois roteadores (produto e cenário).

**O que explicar:** testamos a escolha de **produto** e de **cenário** com perguntas típicas **sem gastar token algum**. É auditável: se errar, sabemos qual palavra-chave ajustar. Erros viram novas palavras-chave nas fichas e nos cenários.

In [ ]:
TESTES_ROTEADOR = [
    ("O cliente quer saber o que é cheque especial", ["cheque_especial"]),
    ("Título descontado vencido e não pago, quem paga?", ["titulos_descontados"]),
    ("Como funciona a antecipação do 13º?", ["antecipacao_13"]),
    ("Cliente quer antecipar a restituição do imposto de renda", ["antecipacao_ir"]),
    ("Qual a diferença entre Price e SACD?", ["credito_pessoal", "credito_garantia_imovel"]),
    ("Posso informar o valor da dívida de um prejuízo?", ["prejuizo"]),
    ("O que posso dizer sobre honra de aval?", ["honra_aval"]),
    ("Cliente é avalista, qual a responsabilidade dele?", ["avalista"]),
    ("Qual programa do crédito rural é para agricultura familiar?", ["credito_rural"]),
    ("Quando é cobrada a tarifa do adiantamento a depositante?", ["adiantamento_depositante"]),
    ("Crédito consignado é para quem?", ["credito_consignado"]),
    ("Cartão em atraso, aceita acordo?", ["cartao"]),
    ("Capital de giro 13º", ["capital_giro", "antecipacao_13"]),
    ("Financiamento de veículo usado", ["financiamento_bens_servicos"]),
    ("Crédito imobiliário com FGTS", ["credito_imobiliario"]),
    ("Como funciona o crédito rotativo?", ["credito_rotativo"]),
    ("Crédito pré-aprovado, aceite pelo celular", ["credito_preaprovado"]),
    ("Empréstimo com garantia de imóvel", ["credito_garantia_imovel"]),
]

TESTES_CENARIOS = [
    ("Qual o prazo do acordo?", "prazo_acordo"),
    ("Posso parcelar em 24 vezes?", "parcelamento"),
    ("Quanto de desconto eu posso dar?", "desconto"),
    ("O contrato está no jurídico, quem paga as custas processuais?", "juridico_custas"),
    ("Em quanto tempo tira o nome do Serasa?", "negativacao"),
    ("O cliente quebrou o acordo", "quebra_acordo"),
    ("O cliente diz que não tem condição de pagar", "objecoes"),
    ("Pode receber o acordo por Pix?", "formalizacao"),
    ("Cartão: posso parcelar em 18x?", "parcelamento"),
]

ac_prod = 0
for pergunta, esperados in TESTES_ROTEADOR:
    achados = rotear_keywords(pergunta)
    ok = any(e in achados for e in esperados)
    ac_prod += ok
    print(("OK  " if ok else "ERRO"), "| produto |", pergunta, "->", achados)

ac_cen = 0
for pergunta, esperado in TESTES_CENARIOS:
    achados = rotear_temas(pergunta)
    ok = esperado in achados
    ac_cen += ok
    print(("OK  " if ok else "ERRO"), "| cenário |", pergunta, "->", achados)

print(f"\nProdutos: {ac_prod}/{len(TESTES_ROTEADOR)} | Cenários: {ac_cen}/{len(TESTES_CENARIOS)}")

## Célula 8 — Instruções do assistente e montagem do contexto

**O que fazer:** rodar a célula.

**O que explicar:** aqui está a parte mais importante para o gestor:

- **`SYSTEM`** define o comportamento: responder **só** com a documentação e as regras fornecidas, **nunca inventar** desconto, prazo, parcela ou valor de custas, respeitar o "pode/não pode informar", **encaminhar ao jurídico/gestão** quando a questão for jurídica ou estiver fora da política, e dar uma **sugestão de fala** em tom de atendimento.
- **`montar_contexto`** monta o pacote enviado ao modelo em cada pergunta: ficha do produto + regras + política de acordo (marcada como fictícia) + regras do cenário + glossário. Se a pergunta não cita produto (ex.: "qual o prazo do acordo?"), entra um **resumo compacto da política de todos os produtos**. **Só o necessário**, por isso o custo é baixo.

In [ ]:
SYSTEM = '''Você é o Assistente de Cobrança, apoio interno para operadores de cobrança do Sicoob.

REGRAS:
1. Responda SOMENTE com base no CONTEXTO recebido. Não use conhecimento externo.
2. Se a informação não estiver no contexto, diga: "Não encontrei isso na documentação; acione a gestão."
3. Sobre acordos, prazos, parcelamento, desconto e cenários de atendimento, use apenas a POLÍTICA DE ACORDOS e os CENÁRIOS DE ATENDIMENTO do contexto. São EXEMPLOS FICTÍCIOS para demonstração, e as regras reais variam por cooperativa. Nunca invente descontos, prazos, parcelas, percentuais ou valores. Se o pedido ultrapassar a política, diga que não é possível e encaminhe à gestão.
4. Se houver REGRA DE INFORMAÇÃO (pode / não pode informar), respeite-a estritamente, inclusive na sugestão de fala.
5. Quando perguntarem sobre saldo ou dívida de um cliente, explique que você não tem acesso a dados de clientes.
6. Questões jurídicas (cobrança judicial, custas processuais, honorários): não calcule valores nem prometa isenção; oriente o operador a acionar o jurídico/gestão.
7. Se a pergunta não citar o produto e a resposta depender dele, peça ao operador para informar o produto, e use o resumo da política apenas como orientação geral.
8. Seja direto e cordial, com no máximo cerca de 150 palavras.
9. Formato: responda diretamente ao operador, sem rótulos como "Resposta ao operador:". Quando útil, finalize em um novo parágrafo com "Sugestão de fala ao cliente:" (1 a 2 frases em tom de atendimento, sem pressionar nem ameaçar o cliente). Não use emojis nem formatação decorativa.
10. Nunca peça nem mostre dados pessoais de clientes.'''


def montar_contexto(ids, pergunta="", temas=None):
    temas = temas or []
    partes = []
    for pid in ids:
        f = FICHAS[pid]
        bloco = [f"### {f['nome']} ({f['categoria']})",
                 f"Atualização de valor: {f['atualizacao']}",
                 f"Descrição: {f['descricao']}"]
        for a in f["atencao"]:
            bloco.append(f"Atenção: {a}")
        if pid in REGRAS_INFORMAR:
            r = REGRAS_INFORMAR[pid]
            bloco.append("REGRA DE INFORMAÇÃO -> PODE informar: " + "; ".join(r["pode"]) +
                         " | NÃO PODE informar: " + "; ".join(r["nao_pode"]))
        p = POLITICA_ACORDOS.get(pid, POLITICA_PADRAO)
        bloco.append("POLÍTICA DE ACORDOS [EXEMPLO FICTÍCIO]: " + formatar_politica(p))
        partes.append("\n".join(bloco))

    if temas:
        linhas = [f"- {TEMAS[t]['nome']}: {TEMAS[t]['regra']}" for t in temas]
        partes.append("### Cenários de atendimento [EXEMPLO FICTÍCIO]\n" + "\n".join(linhas))
        if not ids and any(TEMAS[t]["usa_tabela"] for t in temas):
            partes.append("### Política de acordos por produto [EXEMPLO FICTÍCIO]\n" + resumo_politica())

    if ids:
        texto_norm = normalizar(pergunta + " " + " ".join(FICHAS[i]["descricao"] for i in ids))
        termos = [f"- {t.upper()}: {g['definicao']}" for t, g in GLOSSARIO.items()
                  if any(tem_palavra(texto_norm, k) for k in g["keywords"])]
        if termos:
            partes.append("### Glossário\n" + "\n".join(termos))

    if not partes:
        return "Nenhuma ficha encontrada para esta pergunta."
    return "\n\n".join(partes)


print(montar_contexto(["cartao"], "posso parcelar o cartão em 18x?", ["parcelamento"]))

## Célula 9 — Função principal `responder()`

**O que fazer:** rodar a célula.

**O que explicar:** é o fluxo completo em uma função:

1. **Identifica o cenário** (prazo, parcelamento, custas...) e o **produto**, ambos pelos roteadores (grátis). Se a pergunta for uma continuação ("e dá para parcelar?"), procura o produto nas mensagens anteriores. Só se não achar produto **nem** cenário, usa uma chamada mínima ao modelo (poucos tokens) para classificar.
2. **Monta o contexto** da célula anterior.
3. **Envia ao modelo** com histórico curto (`MAX_HISTORICO`) e resposta limitada (`MAX_TOKENS_RESPOSTA`).
4. **Retorna a resposta e a contagem de tokens**, que permite mostrar o custo real por pergunta.

In [ ]:
MARCA = "\n\n---\n"   # separa o rodapé de debug do texto da resposta


def texto_de(conteudo):
    # histórico do Gradio pode vir como texto ou lista; devolve só o texto, sem rodapé
    if isinstance(conteudo, list):
        conteudo = " ".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in conteudo)
    return str(conteudo).split(MARCA)[0]


def classificar_com_llm(pergunta):
    # plano B: só roda se nenhum roteador por palavras-chave achar nada
    lista = "\n".join(f"{pid}: {f['nome']}" for pid, f in FICHAS.items())
    resp = chamar_modelo(
        "Responda APENAS com o id do produto mais relacionado à pergunta, ou 'nenhum'.",
        [{"role": "user", "content": f"Produtos:\n{lista}\n\nPergunta: {pergunta}"}],
        20,
    )[0].strip().lower()
    return [resp] if resp in FICHAS else []


def identificar_produtos(pergunta, historico, permitir_llm=True):
    ids = rotear_keywords(pergunta)
    if ids:
        return ids, "palavras-chave"
    for m in reversed(historico):                       # pergunta de continuação
        if m.get("role") == "user":
            ids = rotear_keywords(texto_de(m["content"]))
            if ids:
                return ids, "histórico"
    if not permitir_llm:                                # só cenário, sem produto: não gasta token
        return [], "só cenário"
    ids = classificar_com_llm(pergunta)
    return ids, ("classificador LLM" if ids else "não encontrado")


def responder(pergunta, historico=None):
    historico = historico or []
    temas = rotear_temas(pergunta)
    ids, origem = identificar_produtos(pergunta, historico, permitir_llm=not temas)
    contexto = montar_contexto(ids, pergunta, temas)

    msgs = [{"role": m["role"], "content": texto_de(m["content"])} for m in historico[-MAX_HISTORICO:]]
    while msgs and msgs[0]["role"] != "user":           # a API exige começar por 'user'
        msgs.pop(0)
    msgs.append({"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPERGUNTA DO OPERADOR: {pergunta}"})

    texto, t_in, t_out = chamar_modelo(SYSTEM, msgs, MAX_TOKENS_RESPOSTA)
    meta = {"produtos": ids, "temas": temas, "origem": origem,
            "tokens_in": t_in, "tokens_out": t_out}
    return texto, meta

## Célula 10 — Primeiro teste (uma pergunta, com o prompt à vista)

**O que fazer:** rodar e trocar a pergunta para testar variações.

**O que explicar:** mostramos **exatamente o que foi enviado ao modelo** (o contexto) e o consumo de tokens. É a transparência que o gestor precisa: a resposta vem da ficha, e não da "cabeça" do modelo.

In [ ]:
pergunta = "Cliente com cartão em atraso quer parcelar em 18 vezes. Pode?"

temas = rotear_temas(pergunta)
ids, origem = identificar_produtos(pergunta, [], permitir_llm=not temas)
print("Produto(s):", ids, "| origem:", origem, "| Cenário(s):", temas)
print("\n===== CONTEXTO ENVIADO AO MODELO =====")
print(montar_contexto(ids, pergunta, temas))

resposta, meta = responder(pergunta)
print("\n===== RESPOSTA =====")
print(resposta)
print("\nTokens -> entrada:", meta["tokens_in"], "| saída:", meta["tokens_out"])

## Célula 11 — Bateria de testes de ponta a ponta

**O que fazer:** rodar (gasta poucos tokens) e conferir os `OK`/`ATENÇÃO`.

**O que explicar:** são as perguntas de validação, cada uma com **o que a resposta deve conter** e **o que nunca pode aparecer**. Cobrem produtos do PDF e os cenários de call center (prazo, parcelamento acima do limite, custas processuais, quebra de acordo, cliente sem condição, restrição cadastral, Pix em conta pessoal). Dois casos são críticos: *Prejuízo* não pode citar valores em reais, e o pedido de **parcelar acima do máximo** não pode ser aceito.

> A verificação por palavra é simples; leia também as respostas. Para uso real, evoluímos para uma avaliação mais completa.

In [ ]:
TESTES_FINAIS = [
    # (pergunta, termos que DEVEM aparecer, termos que NÃO podem aparecer)
    # --- produtos do PDF ---
    ("Título descontado vencido e não pago: quem responde pela quitação?", ["cooperado"], []),
    ("Posso informar o valor da dívida do cliente em prejuízo?", ["não"], ["R$"]),
    ("O que posso informar sobre honra de aval?", ["vencimento", "atraso"], ["R$"]),
    ("Quando é cobrada a tarifa do adiantamento a depositante?", ["mês seguinte"], []),
    ("Qual a diferença entre Price e SACD?", ["iguais", "decrescentes"], []),
    ("Qual é a taxa de juros do crédito pessoal hoje?", ["não encontrei"], []),   # fora da base
    # --- cenários de call center (política fictícia) ---
    ("O cliente quer parcelar o cartão em 100x com 80% de desconto", ["12", "20"], []),
    ("Qual o prazo do acordo do cartão?", ["10"], []),
    ("Posso parcelar o cheque especial em 36 vezes?", ["24"], []),
    ("Crédito consignado aceita acordo?", ["consign"], []),
    ("O contrato está em cobrança judicial. Quanto são as custas processuais?", ["jurídico"], []),
    ("O cliente quebrou o acordo e atrasou duas parcelas. O que acontece?", ["cancel"], []),
    ("Cliente diz que não tem condição de pagar a entrada. Como contorno?", ["entrada"], []),
    ("Depois do pagamento, em quanto tempo tira o nome do cliente do Serasa?", ["5"], []),
    ("Posso receber o acordo no Pix da minha conta pessoal?", ["não"], []),
    ("Qual o prazo do acordo?", ["prazo"], []),                                      # sem produto
]

total_in = total_out = 0
for pergunta, deve, nao_pode in TESTES_FINAIS:
    resposta, meta = responder(pergunta)
    r_norm = resposta.lower()
    ok = all(t.lower() in r_norm for t in deve) and not any(t.lower() in r_norm for t in nao_pode)
    total_in += meta["tokens_in"]
    total_out += meta["tokens_out"]
    print(("OK       " if ok else "ATENÇÃO  "), "|", pergunta, "|", meta["produtos"], meta["temas"])
    print("   ", resposta.replace("\n", "\n    "))
    print()

n = len(TESTES_FINAIS)
print(f"Tokens médios por pergunta -> entrada: {total_in // n} | saída: {total_out // n}")

## Célula 12 — Página de chat com link para o gestor

**O que fazer:** rodar. O Gradio gera um **link público temporário** (`https://....gradio.live`) para enviar ao gestor.

**O que explicar:** a "página customizada" da demo. O gestor abre o link, digita como operador e vê a resposta, o produto identificado e os tokens gastos (rodapé; desligue com `MOSTRAR_DEBUG = False`).

**Limitações da demo:** o link só funciona **enquanto este notebook estiver rodando** e expira depois de algum tempo. Quem abrir o link usa a **sua** chave de API (custo baixo, mas é seu). Para uso real, publicamos em um servidor próprio.

In [ ]:
import gradio as gr
import inspect

MOSTRAR_FONTE = True   # True = mostra, em letra pequena, qual produto/cenário foi consultado. False = visual limpo.

# Paleta inspirada no verde/turquesa do Sicoob (apenas cores; sem logo ou marca)
VERDE_ESCURO = "#003641"
TURQUESA = "#00AE9D"
VERDE_CLARO = "#7DB61C"
FUNDO = "#F3F8F7"


def chat(mensagem, historico):
    try:
        texto, meta = responder(mensagem, historico)
        if MOSTRAR_FONTE:
            produtos = ", ".join(FICHAS[p]["nome"] for p in meta["produtos"]) or "geral"
            cenarios = ", ".join(TEMAS[t]["nome"] for t in meta["temas"])
            fonte = "Base consultada: " + produtos + (" · Cenário: " + cenarios if cenarios else "")
            texto += MARCA + "<sub>" + fonte + "</sub>"
        return texto
    except Exception as e:
        import traceback
        traceback.print_exc()
        return ("Não consegui responder agora. Tente novamente em instantes.\n\n<sub>Detalhe técnico: " + type(e).__name__ + "</sub>")


CSS = f"""
.gradio-container {{ background: {FUNDO} !important; max-width: 1000px !important; margin: auto !important; }}
#topo {{ background: linear-gradient(120deg, {VERDE_ESCURO} 0%, #00576a 55%, {TURQUESA} 100%);
        border-radius: 16px; padding: 22px 26px; color: white; margin-bottom: 12px; }}
#topo h1 {{ margin: 0 0 4px 0; font-size: 26px; font-weight: 700; color: white; }}
#topo p {{ margin: 0; opacity: .92; font-size: 14px; color: white; }}
#topo .selo {{ display: inline-block; background: {VERDE_CLARO}; color: {VERDE_ESCURO}; font-weight: 700;
              font-size: 11px; letter-spacing: .6px; padding: 3px 10px; border-radius: 999px; margin-bottom: 10px; }}
#rodape {{ text-align: center; color: #54727a; font-size: 12px; margin-top: 10px; }}
#rodape .assina {{ display: block; margin-top: 8px; font-size: 15px; font-weight: 600; color: {TURQUESA}; }}
footer {{ display: none !important; }}
"""

CABECALHO = """
<div id="topo">
  <span class="selo">DEMO INTERNO · DADOS FICTÍCIOS</span>
  <h1>Assistente de Cobrança</h1>
  <p>Tire dúvidas sobre produtos e situações de atendimento. Produtos baseados no material "Produtos Cobrança".
  Regras de acordo, prazo, parcelamento, desconto e jurídico são exemplos fictícios para teste.</p>
</div>
"""

RODAPE = """
<div id="rodape">Protótipo de demonstração · não substitui a orientação da gestão · respostas podem conter imprecisões
  <span class="assina">Suzana ♥</span>
</div>
"""

tema = gr.themes.Base(
    primary_hue=gr.themes.colors.teal,
    neutral_hue=gr.themes.colors.slate,
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
    radius_size=gr.themes.sizes.radius_lg,
).set(
    button_primary_background_fill=TURQUESA,
    button_primary_background_fill_hover=VERDE_ESCURO,
    button_primary_text_color="white",
    body_background_fill=FUNDO,
)

EXEMPLOS = [
    "Cliente com cartão em atraso quer parcelar em 18 vezes. Pode?",
    "Qual o prazo do acordo do cartão?",
    "O contrato está em cobrança judicial. Quem paga as custas processuais?",
    "O cliente quebrou o acordo, o que acontece?",
    "Cliente diz que não tem condição de pagar a entrada. Como oriento?",
    "Em quanto tempo o nome sai do Serasa depois do pagamento?",
    "Posso informar o valor da dívida de um prejuízo?",
    "Título descontado vencido: quem paga?",
]

# Versões novas do Gradio recebem tema/css no launch(); versões antigas, no Blocks()
_novo = "theme" in inspect.signature(gr.Blocks.launch).parameters
with (gr.Blocks() if _novo else gr.Blocks(theme=tema, css=CSS)) as demo:
    gr.HTML(CABECALHO)
    gr.ChatInterface(fn=chat, examples=EXEMPLOS, chatbot=gr.Chatbot(height=480, label="Atendimento"))
    gr.HTML(RODAPE)

if _novo:
    demo.launch(share=True, theme=tema, css=CSS)
else:
    demo.launch(share=True)


## Célula 13 — Roteiro de demonstração, limitações e próximos passos

**Roteiro sugerido para mostrar aos gestores** (nessa ordem, cada um destaca algo diferente):

| # | Pergunta no chat | O que mostrar |
|---|---|---|
| 1 | "Título descontado vencido: quem paga?" | Resposta fiel ao material do PDF. |
| 2 | "Posso informar o valor da dívida de um prejuízo?" | Regra "pode/não pode informar" respeitada. |
| 3 | "Cliente com cartão em atraso quer parcelar em 18 vezes. Pode?" | Respeita o limite da política (12x) e não promete além. |
| 4 | "Qual o prazo do acordo?" (sem produto) | O chat pede o produto e dá orientação geral. |
| 5 | "O contrato está em cobrança judicial. Quem paga as custas?" | Não calcula custas e encaminha ao jurídico. |
| 6 | "O cliente quebrou o acordo, o que acontece?" | Cenário pós-acordo. |
| 7 | "Cliente diz que não tem condição de pagar a entrada" | Alternativas dentro da política e sugestão de fala empática. |
| 8 | "Qual a taxa de juros do crédito pessoal hoje?" | Admite que não sabe e aciona a gestão (sem inventar). |

**Limitações conhecidas**
- **Política de acordos, prazos, parcelamento, desconto, restrição cadastral e jurídico/custas são fictícios.** O que for real precisa vir da gestão e do jurídico, por cooperativa.
- O material tem **inconsistências** que o assistente repete: "SACD" é descrito como "Constante *Decrescente*" (o nome usual é SAC); o slide de Financiamento de Bens e Serviços não informa a regra de atualização; Honras de Avais e Prejuízo não têm descrição, só a regra de informação.
- O assistente **não consulta dados de clientes** (de propósito, por LGPD) e **não calcula** saldos nem custas.
- Os roteadores por palavras-chave podem errar com sinônimos; há plano B com o modelo para produto, mas não para cenário.

**Próximos passos**
1. Validar as fichas com a área de cobrança e corrigir o texto-fonte.
2. Substituir a **política fictícia** pela real, por cooperativa (planilha ou sistema).
3. Revisar com o jurídico o cenário de custas e honorários.
4. Ampliar os testes com casos reais anonimizados.
5. Se a base crescer (várias cooperativas, normativos, contratos), adicionar **RAG** (busca vetorial) por cima desta estrutura.
6. Publicar em ambiente próprio, com controle de acesso e registro de uso.